# Assessment Workflow Orchestrator

**Purpose:** Orchestrates the complete assessment PDF ingestion workflow
- **Step 1:** Sync PDFs from OneDrive/SharePoint to Fabric Lakehouse
- **Step 2:** Parse PDFs and ingest into Delta tables
- **Step 3:** Validate data quality and send notifications

**Schedule:** Run daily at 2:00 AM UTC (configurable)

**Default lakehouse:** ManagedServiceData

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

from datetime import datetime, timezone
import json

# Workflow settings
WORKFLOW_CONFIG = {
    "sync_enabled": True,
    "parse_enabled": True,
    "validation_enabled": True,
    "notification_enabled": False,  # Set to True to enable email notifications
    "notification_email": "admin@example.com",  # Update with your email
}

# OneDrive sync settings (same as onedrive_to_fabric_sync.py)
ONEDRIVE_CONFIG = {
    "client_id": "your-client-id-here",  # Update from Azure App Registration
    "tenant_id": "your-tenant-id-here",  # Update from Azure AD
    "scopes": ["https://graph.microsoft.com/.default"],
    "source_folder_id": "your-onedrive-folder-id",  # OneDrive/SharePoint folder ID
    "lakehouse_base_path": "Files",
}

print("✅ Configuration loaded")
print(f"   Sync enabled: {WORKFLOW_CONFIG['sync_enabled']}")
print(f"   Parse enabled: {WORKFLOW_CONFIG['parse_enabled']}")
print(f"   Validation enabled: {WORKFLOW_CONFIG['validation_enabled']}")

In [ ]:
# ============================================================================
# STEP 1: SYNC PDFs FROM ONEDRIVE TO LAKEHOUSE
# ============================================================================

if WORKFLOW_CONFIG["sync_enabled"]:
    print("\n" + "=" * 80)
    print("STEP 1: SYNCING PDFs FROM ONEDRIVE")
    print("=" * 80 + "\n")
    
    import requests
    from msal import PublicClientApplication
    import notebookutils
    from datetime import date
    
    # Authentication
    def get_access_token():
        app = PublicClientApplication(
            client_id=ONEDRIVE_CONFIG["client_id"],
            authority=f"https://login.microsoftonline.com/{ONEDRIVE_CONFIG['tenant_id']}"
        )
        
        result = app.acquire_token_interactive(scopes=ONEDRIVE_CONFIG["scopes"])
        
        if "access_token" in result:
            return result["access_token"]
        else:
            raise Exception(f"Authentication failed: {result.get('error_description')}")
    
    def list_files_in_folder(folder_id, access_token):
        url = f"https://graph.microsoft.com/v1.0/me/drive/items/{folder_id}/children"
        headers = {"Authorization": f"Bearer {access_token}"}
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return response.json().get("value", [])
    
    def download_file(file_id, access_token):
        url = f"https://graph.microsoft.com/v1.0/me/drive/items/{file_id}/content"
        headers = {"Authorization": f"Bearer {access_token}"}
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return response.content
    
    def categorize_file(filename):
        filename_lower = filename.lower()
        if "security" in filename_lower and "assessment" in filename_lower:
            return "security_assessment"
        elif "copilot" in filename_lower and "readiness" in filename_lower:
            return "copilot_readiness"
        elif "copilot" in filename_lower and "assessment" in filename_lower:
            return "copilot_assessment"
        return None
    
    def get_dated_folder(base_folder, category):
        today = date.today().strftime('%Y-%m-%d')
        return f"{base_folder}/{category}/{today}"
    
    def upload_to_fabric(file_content, target_path):
        try:
            notebookutils.fs.put(target_path, file_content.decode('latin-1'), overwrite=True)
            return "success"
        except Exception as e:
            print(f"      ❌ Upload failed: {e}")
            return "failed"
    
    # Run sync
    try:
        token = get_access_token()
        files = list_files_in_folder(ONEDRIVE_CONFIG["source_folder_id"], token)
        
        stats = {"total": 0, "uploaded": 0, "skipped": 0, "failed": 0}
        
        for file in files:
            if not file["name"].lower().endswith(".pdf"):
                continue
            
            stats["total"] += 1
            category = categorize_file(file["name"])
            
            if not category:
                print(f"   ⚠️ Skipping {file['name']} (unknown category)")
                stats["skipped"] += 1
                continue
            
            target_folder = get_dated_folder(ONEDRIVE_CONFIG["lakehouse_base_path"], category)
            target_path = f"{target_folder}/{file['name']}"
            
            # Check if file exists
            try:
                notebookutils.fs.head(target_path, 1)
                print(f"   ⏭️ {file['name']} → already exists")
                stats["skipped"] += 1
                continue
            except:
                pass  # File doesn't exist, proceed with upload
            
            # Download and upload
            content = download_file(file["id"], token)
            result = upload_to_fabric(content, target_path)
            
            if result == "success":
                print(f"   ✅ {file['name']} → {target_path}")
                stats["uploaded"] += 1
            else:
                stats["failed"] += 1
        
        print(f"\n📊 Sync Summary:")
        print(f"   Total files: {stats['total']}")
        print(f"   Uploaded: {stats['uploaded']}")
        print(f"   Skipped: {stats['skipped']}")
        print(f"   Failed: {stats['failed']}")
        
        sync_success = stats["failed"] == 0
        
    except Exception as e:
        print(f"❌ Sync failed: {e}")
        sync_success = False
else:
    print("⏭️ Sync step skipped (disabled in config)")
    sync_success = True

In [ ]:
# ============================================================================
# STEP 2: PARSE PDFs AND INGEST INTO DELTA TABLES
# ============================================================================

if WORKFLOW_CONFIG["parse_enabled"] and sync_success:
    print("\n" + "=" * 80)
    print("STEP 2: PARSING PDFs AND INGESTING DATA")
    print("=" * 80 + "\n")
    
    # Run the parse_assessment_pdfs notebook
    try:
        result = notebookutils.notebook.run(
            "parse_assessment_pdfs",
            timeoutSeconds=3600,  # 1 hour timeout
            arguments={}
        )
        
        print("✅ PDF parsing completed successfully")
        print(f"   Result: {result}")
        parse_success = True
        
    except Exception as e:
        print(f"❌ PDF parsing failed: {e}")
        parse_success = False
else:
    if not WORKFLOW_CONFIG["parse_enabled"]:
        print("⏭️ Parse step skipped (disabled in config)")
    elif not sync_success:
        print("⏭️ Parse step skipped (sync failed)")
    parse_success = False

In [ ]:
# ============================================================================
# STEP 3: VALIDATE DATA QUALITY
# ============================================================================

if WORKFLOW_CONFIG["validation_enabled"] and parse_success:
    print("\n" + "=" * 80)
    print("STEP 3: VALIDATING DATA QUALITY")
    print("=" * 80 + "\n")
    
    validation_results = {}
    
    # Check Security Assessment Reports
    try:
        result = spark.sql("""
            SELECT 
                COUNT(*) as total,
                COUNT(tenant_name) as has_tenant,
                COUNT(assessment_date) as has_date,
                COUNT(overall_score_percentage) as has_score
            FROM security_assessment_reports
        """).collect()[0]
        
        validation_results["security_reports"] = {
            "total": result.total,
            "completeness": (result.has_tenant / result.total * 100) if result.total > 0 else 0
        }
        print(f"✓ Security Reports: {result.total} records, {validation_results['security_reports']['completeness']:.1f}% complete")
    except Exception as e:
        print(f"✗ Security Reports validation failed: {e}")
        validation_results["security_reports"] = {"error": str(e)}
    
    # Check Copilot Readiness Reports
    try:
        result = spark.sql("""
            SELECT 
                COUNT(*) as total,
                COUNT(organisation_name) as has_org,
                COUNT(assessment_date) as has_date
            FROM copilot_readiness_reports
        """).collect()[0]
        
        validation_results["copilot_readiness"] = {
            "total": result.total,
            "completeness": (result.has_org / result.total * 100) if result.total > 0 else 0
        }
        print(f"✓ Copilot Readiness: {result.total} records, {validation_results['copilot_readiness']['completeness']:.1f}% complete")
    except Exception as e:
        print(f"✗ Copilot Readiness validation failed: {e}")
        validation_results["copilot_readiness"] = {"error": str(e)}
    
    # Check Copilot Assessment Reports
    try:
        result = spark.sql("""
            SELECT 
                COUNT(*) as total,
                COUNT(organisation_name) as has_org,
                COUNT(overall_score_percentage) as has_score
            FROM copilot_assessment_reports
        """).collect()[0]
        
        validation_results["copilot_assessment"] = {
            "total": result.total,
            "completeness": (result.has_org / result.total * 100) if result.total > 0 else 0
        }
        print(f"✓ Copilot Assessment: {result.total} records, {validation_results['copilot_assessment']['completeness']:.1f}% complete")
    except Exception as e:
        print(f"✗ Copilot Assessment validation failed: {e}")
        validation_results["copilot_assessment"] = {"error": str(e)}
    
    validation_success = True
else:
    print("⏭️ Validation step skipped")
    validation_success = False
    validation_results = {}

In [ ]:
# ============================================================================
# STEP 4: WORKFLOW SUMMARY & NOTIFICATIONS
# ============================================================================

print("\n" + "=" * 80)
print("WORKFLOW SUMMARY")
print("=" * 80 + "\n")

workflow_end = datetime.now(timezone.utc)
workflow_status = "SUCCESS" if (sync_success and parse_success) else "FAILED"

summary = {
    "workflow_status": workflow_status,
    "timestamp": workflow_end.isoformat(),
    "steps": {
        "sync": "✅ Success" if sync_success else "❌ Failed",
        "parse": "✅ Success" if parse_success else "❌ Failed",
        "validation": "✅ Success" if validation_success else "⏭️ Skipped"
    },
    "validation_results": validation_results
}

print(f"Status: {workflow_status}")
print(f"Timestamp: {workflow_end}")
print(f"\nStep Results:")
for step, status in summary["steps"].items():
    print(f"  {step.capitalize()}: {status}")

if validation_results:
    print(f"\nData Quality:")
    for table, metrics in validation_results.items():
        if "error" not in metrics:
            print(f"  {table}: {metrics['total']} records ({metrics['completeness']:.1f}% complete)")

# Store summary in lakehouse for tracking
try:
    summary_json = json.dumps(summary, indent=2)
    summary_path = f"Files/workflow_logs/{workflow_end.strftime('%Y-%m-%d_%H-%M-%S')}_summary.json"
    notebookutils.fs.put(summary_path, summary_json, overwrite=True)
    print(f"\n📁 Summary saved to: {summary_path}")
except Exception as e:
    print(f"\n⚠️ Failed to save summary: {e}")

# Send notification (if enabled)
if WORKFLOW_CONFIG["notification_enabled"]:
    print("\n📧 Sending notification email...")
    # TODO: Implement email notification using SendGrid, Azure Logic Apps, or similar
    print("   (Email notification not yet implemented)")

print("\n" + "=" * 80)
print(f"✅ WORKFLOW COMPLETE: {workflow_status}")
print("=" * 80)